<a href="https://colab.research.google.com/github/updp16/Research_Appendices/blob/main/Chlorophyll_a_feature_extraction_and_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Feature Extraction and correlation analysis(Pearson correlation)**

In [ ]:
import pandas as pd
import itertools
import numpy as np

df = pd.read_csv(r"https://raw.githubusercontent.com/updp16/research/refs/heads/main/merged_data_ShellPoint_test.csv")

bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8','B8A','B11','B12']

new_features = {}

for b1, b2 in itertools.permutations(bands, 2):
    # Band ratio
    new_features[f'RATIO_{b1}_{b2}'] = df[b1] / df[b2]

    # Normalized difference
    denominator = df[b1] + df[b2]
    new_features[f'ND_{b1}_{b2}'] = (df[b1] - df[b2]) / denominator

df = pd.concat([df, pd.DataFrame(new_features)], axis=1)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print(df.head())



print("\nPearson correlation:")
corr = df.corr(numeric_only=True)['Actual'].sort_values(ascending= False)

print(corr.head(20))

# df['Actual'].corr(df['ND_B4_B5'])

from google.colab import files

df.to_csv("Extracted_features_Chlorophyll.csv", index=False)

# files.download("Extracted_features_Chlorophyll.csv")


   Actual  B11  B12    B2    B3    B4    B5   B6   B7   B8  ...  RATIO_B12_B6  \
1    1.92  860  750   982  1104  1045  1017  834  819  826  ...      0.899281   
2    3.93  873  768  1001   994  1002  1002  900  802  837  ...      0.853333   
3    2.83  674  600   628   697   758   849  741  705  577  ...      0.809717   
4    5.86   57   67     7    47    69    99   16   34   31  ...      4.187500   
9    1.65  595  522   799   997   840   764  587  593  619  ...      0.889267   

   ND_B12_B6  RATIO_B12_B7  ND_B12_B7  RATIO_B12_B8  ND_B12_B8  RATIO_B12_B8A  \
1  -0.053030      0.915751  -0.043977      0.907990  -0.048223       0.912409   
2  -0.079137      0.957606  -0.021656      0.917563  -0.042991       0.924188   
3  -0.105145      0.851064  -0.080460      1.039861   0.019541       0.877193   
4   0.614458      1.970588   0.326733      2.161290   0.367347       3.722222   
9  -0.058611      0.880270  -0.063677      0.843296  -0.085013       0.942238   

   ND_B12_B8A  RATIO_B12_B

# **Feature Selection (mRMR algorithm)**


In [ ]:
!pip install mrmr-selection

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Load dataset
# ==========================================
df = pd.read_csv(r"https://raw.githubusercontent.com/updp16/research/refs/heads/main/Extracted_features_Chlorophyll.csv")

# ==========================================
# 2. Separate features and target
# ==========================================
target_col = "Actual"

X = df.drop(columns=[target_col])
y = df[target_col]

# ==========================================
# 3. Clean data
# ==========================================
X = X.replace([np.inf, -np.inf], np.nan)

# Combine and drop rows with missing values
data = pd.concat([X, y], axis=1).dropna()

X = data.drop(columns=[target_col])
y = data[target_col]

# ==========================================
# 4. Remove non-numeric columns if any
# ==========================================
X = X.select_dtypes(include=[np.number])

# ==========================================
# 5. Remove constant columns
# ==========================================
constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
X = X.drop(columns=constant_cols)

# ==========================================
# 6. Run mRMR feature selection
# ==========================================
from mrmr import mrmr_regression

# Choose how many features you want
K = 10

selected_features = mrmr_regression(X=X, y=y, K=K)

print("\nSelected Features by mRMR:")
print(selected_features)

# ==========================================
# 7. Show selected dataset
# ==========================================
X_selected = X[selected_features]

print("\nShape of selected feature set:")
print(X_selected.shape)

# print("\nPreview of selected features:")
# print(X_selected.head())

100%|██████████| 10/10 [00:00<00:00, 20.24it/s]


Selected Features by mRMR:
['RATIO_B5_B4', 'RATIO_B2_B4', 'ND_B5_B12', 'ND_B3_B5', 'ND_B5_B3', 'RATIO_B5_B3', 'ND_B4_B5', 'ND_B5_B4', 'ND_B3_B4', 'ND_B4_B3']

Shape of selected feature set:
(104, 10)
